In [ ]:
from ..database.init_db import init_db
from fastapi import FastAPI, UploadFile, File, Depends, HTTPException
from sqlalchemy.orm import Session
import json

from dotenv import load_dotenv
load_dotenv() 
import sys
import os
from dateutil import parser

from typing import List
from ..models.models import Candidate, Education, Experience, Certification, Skill, Language
from ..models.candidate_schema import CandidateIn, CandidateOut
from ..services.info_extract import extract_text_from_pdf, extract_info
from vector_db import CandidateFaissIndex
from database import init_db, SessionLocal 
from datetime import datetime, date

In [ ]:
init_db()

# Database for production
faiss_index = None  
EMBED_DIM = 1536

# Procsess to handle None or empty values safely
def safe_get(value, default="Unknown"):
    if value is None:
        return default
    if isinstance(value, str) and value.strip() == "":
        return default
    return value

def parse_date(date_str):
    current_dt = datetime.now()

    if not isinstance(date_str, str) or not date_str.strip():
        return "Unknown"

    cleaned_date_str = date_str.strip().lower()

    # Handle common non-date strings
    if cleaned_date_str in ("present", "current", "now", "na", "n/a", "null", "", "unknown"):
        return "Unknown"
    
    try:
        parsed_dt = parser.parse(date_str.strip(), default=current_dt)
        return parsed_dt.strftime("%Y-%m-%d")
    except (ValueError, TypeError, parser.ParserError):
        return "Unknown"
    except Exception:
        return "Unknown"


# Database session dependency
def get_db():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()

def create_or_get_skill(db, name):
    obj = db.query(Skill).filter(Skill.name == name).first()
    if not obj:
        obj = Skill(name=name)
        db.add(obj)
        db.commit()
        db.refresh(obj)
    return obj

def create_or_get_language(db, name):
    obj = db.query(Language).filter(Language.name == name).first()
    if not obj:
        obj = Language(name=name)
        db.add(obj)
        db.commit()
        db.refresh(obj)
    return obj
